# Arrays and Hash Tables

Python reference-type semantics as they apply to arrays/lists: mutation vs. rebinding, `+=` vs `+`, `==` vs `is`, passing lists to functions, mutable vs. immutable types, and variable scope (LEGB).

## Reference types

A variable holding a list doesn't hold the list itself — it holds a reference (pointer) to where the list lives in memory.

```python
a = [1, 2, 3]
b = a
b.append(4)
print(a)  # ?
```

`a` and `b` point to the *same* list, so mutating through `b` is visible through `a` too.

Contrast with value types (ints, strings, tuples) — assignment copies the value, so mutating one name never affects the other.

In [ ]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
b.append(4)
print(a,b,c)

b = b + [4]
print(a, b, c)

[1, 2, 3, 4] [1, 2, 3, 4] [1, 2, 3]


## `+=` vs `+` for lists — mutation vs. new object

They look equivalent but aren't, for mutable types like lists:

```python
b += [4]      # calls __iadd__ -> mutates the list in place (like .extend())
b = b + [4]   # calls __add__   -> builds a brand-new list, then rebinds b
```

- `b += [4]`: mutates the existing list object. Any other name pointing at the same list (e.g. `a`) sees the change too.
- `b = b + [4]`: creates a new list at a new memory address, and rebinds `b` to point there. The connection between `a` and `b` is broken — `a` still points to the old address, `b` now points elsewhere. They're no longer aliases of the same object.

For immutable types (int, str, tuple), there's no in-place version, so `+=` and `x = x + y` behave identically.

**Interview signal:** if a question says an operation happens "in place," that means mutation — the original object is modified directly, no new object is created.

In [9]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
print(b == c)
print(a is b)

True
True


## `==` vs `is`, and Python vs Java

- `==` checks **value equality** — do the two objects contain the same data?
- `is` checks **identity** — are the two names pointing to the exact same object in memory (same address)?

```python
a = [1, 2, 3]
b = a
c = [1, 2, 3]

a is b   # True  -> same object (b was assigned from a)
b is c   # False -> different objects, even though...
b == c   # True  -> ...their contents are equal
```

**Python vs Java:** in Python, `==` on lists compares values by default (element-wise). In Java, `==` on arrays compares references (identity) — two arrays with identical elements are `==` false unless they're literally the same object; you need `Arrays.equals()` to compare values. So Python's default list equality is what Java makes you opt into explicitly.

In [11]:
def append_four(x):
    x.append(4)

def add_four(x):
    x = x + [4]

'''
a = [1, 2, 3]
append_four(a)
print(a)  

'''
b = [1, 2, 3]
add_four(b)
print(b) 


[1, 2, 3]


## Passing lists to functions

A function parameter is just another name pointing at the same object the caller passed in — same idea as `b = a`.

- `append_four(a)`: mutates the list in place (`x.append(4)`), so `a` shows the change after the call.
- `add_four(b)`: rebinds the *local* name `x` to a new list (`x = x + [4]`); that only affects `x`'s local frame, so `b` is unchanged after the call.

Rule of thumb: mutating methods on the parameter affect the caller; reassigning the parameter does not.

## Mutable vs. immutable — the real thing that bites people

Everything in Python is a reference — including "value-like" things such as ints. What actually differs is whether the *object* can be changed in place:

- **Mutable** (list, dict, set, most custom class instances): two names can share one object; mutating through one name is visible through the other.
- **Immutable** (int, str, tuple, float, bool, frozenset): the object itself can never change. `x = 6` after `x = 5` doesn't mutate the int `5` — it rebinds `x` to a different object. This *looks* like value semantics, but it's still references underneath; there's just no in-place mutation to observe.

Caveat: tuples are only shallowly immutable — you can't reassign an element, but if a tuple holds a mutable object (e.g. a list), that inner object can still be mutated.

### `x += 5` vs `x += [5]`

```python
x = 5
x += 5   # int has no __iadd__ -> falls back to __add__, creates a NEW int object, rebinds x

x = [5]
x += [5]  # list HAS __iadd__ -> mutates the list in place, same object, same address
```

`x += 5`: since ints are immutable, there's no in-place version — Python creates a new int object and rebinds `x` to it (new memory address). `x += [5]`: lists are mutable and define `__iadd__`, so this mutates the existing list in place (same address, same object as before).

In [15]:
x = "builtin"  # placeholder, not used directly

def outer():
    enclosing_var = "enclosing"

    def inner():
        local_var = "local"
        print(local_var)       # Local
        print(enclosing_var)   # Enclosing
        print(global_var)      # Global
        print(len)             # Builtin

    inner()

global_var = "global"
outer()

print("================")
#print(local_var)       # retuns error
#print(enclosing_var)   # returns error
print(global_var)      # Global
print(len)             # Builtin

local
enclosing
global
<built-in function len>
global
<built-in function len>


## Scope — LEGB

Scope is where a name is visible / which namespace it lives in. Python resolves names via the **LEGB** rule, checked in this order:

- **Local** — inside the current function
- **Enclosing** — an outer (enclosing) function, for nested functions/closures
- **Global** — module level
- **Builtin** — `len`, `print`, etc.

### Global vs. Builtin — why they're separate tiers

Each **module** (each `.py` file) has its own distinct global namespace — `mod_a.py` and `mod_b.py` do not share global variables. **Builtins**, by contrast, live in a single shared namespace (the `builtins` module) that every module implicitly consults — it is not duplicated per module.

If builtins were folded into "global," each module would need its own binding for `len`, `print`, etc., since global scope is module-local. Keeping Builtin as a separate, outermost tier means every module gets the same fallback definitions without redeclaring them, and any module can locally shadow a builtin name (e.g. defining its own `len`) without affecting other modules' access to the original.

## Implementing a dynamic array from scratch

Python's `list` is itself a dynamic array: fixed-size storage under the hood that reallocates (roughly doubles) when it fills up, giving `append` amortized O(1) time. Implementing a simplified version is a classic interview exercise and reinforces the reference/mutation concepts above.

In [51]:
# TODO: Implement a simplified dynamic array class `MyList` that:
# - starts empty, backed by a fixed-size internal array that doubles in capacity when full
# - supports append(value) -> amortized O(1)
# - supports len(mylist) -> number of elements currently stored (not the internal capacity)
# - supports mylist[i] -> get the i-th element
# - supports pop() -> remove and return the last element

class MyList:
    def __init__(self):
        self._length = 0
        self._lst = [None]
    
    def __len__(self):
        return self._length 
    
    def __getitem__(self, index):
        assert index < self._length, "index outside the length!"
        return self._lst[index]
    
    def pop(self):
        last_element = self._lst[self._length - 1]
        self._length -=1
        return last_element
    
    def _grow(self):
        _new_mem = [None] * self._length * 2
        for i in range(self._length):
            _new_mem[i] = self._lst[i]
        self._lst = _new_mem
    
    def append(self, value):
        self._lst[self._length] = value
        self._length +=1
        if len(self._lst) == self._length: 
            self._grow()
    
    def remove(self, index):
        for i in range(self._length):
            if i > index:
                self._lst[i - 1] = self._lst[i]
        self._length -= 1

## Array time complexity

| Operation | Complexity | Why |
|---|---|---|
| Search (find a value) | O(n) | No shortcut — scan until found (unless sorted + binary search) |
| Lookup (access by index) | O(1) | Direct address calculation from the index |
| Append (add to end) | O(1) amortized | No shifting needed; occasional resize/copy averages out over many appends |
| Prepend (add to front) | O(n) | Every existing element has to shift over by one slot |
| Insert (at arbitrary index) | O(n) | Everything after the insertion point has to shift |
| Delete (at arbitrary index) | O(n) | Everything after the deleted element has to shift back |

In [68]:
# TODO: Reverse a string.

# solution 1: 

def reverse(string):
    list_str = list(string)
    n = len(list_str)

    new_list = [None] * n
    for i in range(n):
        new_list[i] = list_str[n-1-i]
    
    return ''.join(new_list)

# solution 2: the problem with solution 1 is the space complecity of o(n)

def reverse2(string):
    list_str = list(string)
    n = len(list_str)

    for i in range(n):
        if i < n//2:
            temp = list_str[i]
            list_str[i] = list_str[n-1-i]
            list_str[n-1-i] = temp
            
    return ''.join(list_str)

print(reverse2('aab'))

baa


In [ ]:
# TODO: Merge two sorted arrays into a single sorted array.

def merge(ls1, ls2):
    n = len(ls1)
    m = len(ls2)
    indx1 = 0
    indx2 = 0
    new_ls = [None] * (n+m)
    for i in range(n+m):
    
        if indx1 < n and indx2 < m: 
            if ls1[indx1] < ls2[indx2]:
                new_ls[i] = ls1[indx1]
                indx1 += 1 
            else:
                new_ls[i] = ls2[indx2]
                indx2 += 1
                
        elif indx1 < n:
            new_ls[i] = ls1[indx1]
            indx1 += 1

        else:
            new_ls[i] = ls2[indx2]
            indx2 += 1
            
    return new_ls

[7, 8, 10]


A **dict** is Python's built-in hash table: a collection of key-value pairs, where each key maps to a value and supports fast lookup/insertion/deletion by key.

## Dict keys: any hashable object — including functions

A dict key isn't limited to strings/numbers — it can be **any hashable object**. That includes functions, since functions are objects too (like everything else in Python) and are hashable by default.

**Hashable** means an object defines a `__hash__` method that returns a consistent integer for that object's lifetime, so it can be run through a hash function and mapped to a slot. Immutable types (int, str, tuple, frozenset) are hashable by default. Functions are also hashable by default — their hash is based on identity (same function object → same hash), not on their code or arguments. Mutable types like `list`, `dict`, and `set` are **not** hashable — since their contents can change, their "identity" for hashing purposes can't stay stable, so Python disallows them as dict keys entirely (`TypeError: unhashable type`).

**What a hash function does:** it takes an object (of arbitrary size/complexity) and deterministically maps it to a fixed-size integer — the *hash value*. That integer is then used (typically via `hash(key) % number_of_slots`) to compute an index into the dict's underlying array, which is the slot the key/value pair gets stored in. The same object always hashes to the same value within a single run, so looking a key up means: hash it, jump straight to that slot, done — no scanning required.

## Why insert/delete is O(1) for dicts but O(n) for lists

**Lists are ordered and contiguous.** Position `i` in a list is a slot that's a reference to where a value lives — and slots are laid out one after another, in order. When you insert or delete somewhere in the **middle**, every element that comes after has to shift to a new slot to keep that contiguous ordering intact. That shifting is what makes insert/delete O(n) — you're not just touching one slot, you're touching all the ones after it.

Note: this only applies to inserting/deleting at an arbitrary position. **Appending to the end is O(1) amortized** (see `MyList.append` above) — nothing needs to shift, since nothing comes after the new last element.

**Dicts have no such ordering to maintain.** A key isn't stored based on its position — it's run through a hash function (see above), and the resulting index is what points to the slot holding the value. Keys don't live in any particular order in memory; where a key's value ends up is essentially arbitrary (determined by its hash). Since there's no contiguous ordering to preserve, inserting or deleting a key never requires moving any other elements around — you just compute the hash, go straight to that slot, and read/write/remove it. That's what makes dict insert/delete O(1) (assuming no hash collision for insert).

In [87]:
# Functions are hashable objects too, so they can be used as dict keys.

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

function_descriptions = {
    add: "adds two numbers",
    subtract: "subtracts the second number from the first",
    multiply: "multiplies two numbers",
}

for func, description in function_descriptions.items():
    print(func.__name__, "->", description)

add -> adds two numbers
subtract -> subtracts the second number from the first
multiply -> multiplies two numbers


## Sets vs. dicts

A **set** is essentially a dict with only keys, no values — same hash table mechanics (hash key → index → slot), just tracking presence rather than storing associated data.

Use a **set** when you need to know "have I seen this before?" — fast O(1) average membership checks, deduplication.

Use a **dict** when you need to associate data *with* each key, not just track its presence — e.g. counting occurrences (key → count), caching/memoization (input → result), grouping (key → list of items), or general lookup tables (id → record).

**Sets are iterable, but not indexable:** you can do `for i in a: ...`, but not `a[0]` — since a set has no positional ordering (elements live wherever their hash puts them), there's no "0th element" to point to.

### Common interview patterns that use dicts

- **Frequency counting** — counting occurrences of characters/words/elements (e.g. anagram checks, most frequent element).
- **Two-sum style lookups** — store `value → index` as you iterate, check if the complement exists, turning O(n²) into O(n).
- **Grouping** — e.g. group anagrams by sorted-letters key, group items by category.
- **Memoization/caching** — store `input → computed result` to avoid recomputation (common in DP problems).
- **Seen-before with extra info** — like a set's "have I seen this," but when you also need to remember *where*/*what* (e.g. first index of a duplicate).
- **Graph adjacency lists** — `node → list of neighbors`.
- **LRU cache** — combining a dict with a doubly linked list for O(1) get/put with eviction order.

In [107]:
# TODO: Implement a hash table from scratch: `SimpleDict`.
# The hash function (key -> bucket index) is already done for you below.
# Collisions are handled via chaining: each bucket is a list of (key, value) pairs.
#
# Implement:
# - d[key] = value       -> __setitem__ (sets or updates)
# - d[key]               -> __getitem__ (raises KeyError if missing)
# - key in d             -> __contains__
# - len(d)               -> __len__
# - del d[key]           -> __delitem__ (raises KeyError if missing)
# - d.get(key, default)  -> like dict.get, returns default if key missing (no KeyError)
# - d.keys()             -> all keys
# - d.values()           -> all values
# - d.items()            -> all (key, value) pairs
# - d.update(other)      -> merge another SimpleDict (or dict) into this one

class SimpleDict:
    def __init__(self, capacity=8):
        self._capacity = capacity
        self._buckets = [[] for _ in range(capacity)]
        self._length = 0

    def _hash_index(self, key):
        return hash(key) % self._capacity

    def __setitem__(self, key, value):
        bucket_indx = self._hash_index(key)
        bucket = self._buckets[bucket_indx]
        for i, (b_key, b_value) in enumerate(bucket):
            if b_key == key:
                bucket[i] = (key, value)   # key exists → overwrite
                return
        bucket.append((key, value))        # not found → add new
        self._length += 1


    def __getitem__(self, key):
        bucket_indx = self._hash_index(key)
        for b_key, b_value in self._buckets[bucket_indx]:
            if b_key == key:
                return b_value    
        raise KeyError(f"{key} not found")

    def __contains__(self, key):
        bucket_indx = self._hash_index(key)
        bucket = self._buckets[bucket_indx]
        for b_key, _ in bucket:
            if b_key == key:
                return True
        return False
        

    def __len__(self):
        return self._length

    def __delitem__(self, key):
        bucket_indx = self._hash_index(key)
        bucket = self._buckets[bucket_indx]
        for i, (b_key, b_value) in enumerate(bucket):
            if b_key == key:
                del bucket[i]
                self._length -= 1
                return
        raise KeyError(f"{key} not found")

    def get(self, key, default=None):
        bucket_indx = self._hash_index(key)
        for b_key, b_value in self._buckets[bucket_indx]:
            if b_key == key:
                return b_value    
        return default 

    def keys(self):
        n_buckets = len(self._buckets)
        keys = []
        # O(n + capacity): outer loop visits all `capacity` buckets once each;
        # inner loop iterations, summed across every bucket, total exactly `n`
        # (every stored pair lives in exactly one bucket) -- not n * capacity.
        for bucket_indx in range(n_buckets): 
            for key, _ in self._buckets[bucket_indx]:
                keys.append(key)
        return keys


    def values(self):
        n_buckets = len(self._buckets)
        values = []
        # O(n + capacity): outer loop visits all `capacity` buckets once each;
        # inner loop iterations, summed across every bucket, total exactly `n`
        # (every stored pair lives in exactly one bucket) -- not n * capacity.
        for bucket_indx in range(n_buckets): 
            for _, value in self._buckets[bucket_indx]:
                values.append(value)
        return values

    def items(self):
        n_buckets = len(self._buckets)
        items = []
        # O(n + capacity): outer loop visits all `capacity` buckets once each;
        # inner loop iterations, summed across every bucket, total exactly `n`
        # (every stored pair lives in exactly one bucket) -- not n * capacity.
        for bucket_indx in range(n_buckets): 
            for key, value in self._buckets[bucket_indx]:
                items.append((key, value))
        return items

    def update(self, other):
        other_items = other.items()
        for key, value in other_items:
            self.__setitem__(key, value)

## Hash table time complexity

| Operation | Average case | Worst case | Why |
|---|---|---|---|
| Search / lookup by key (`key in d`, `d[key]`) | O(1) | O(n) | Hash the key, jump straight to its bucket — but if every key collides into the same bucket (bad hash / adversarial input), that bucket degrades to a linear scan |
| Search by value (no key, just "does this value exist anywhere") | O(n) | O(n) | Only keys are hashed/indexed — values aren't, so there's no shortcut; you must scan every pair regardless of collisions |
| Insert (`d[key] = value`) | O(1) | O(n) | Same reasoning as key lookup — hashing to find the bucket is O(1), but a fully collided bucket means scanning it to check for an existing key before appending |
| Delete (`del d[key]`) | O(1) | O(n) | Same as insert — find the bucket in O(1), but removing from within a badly-collided bucket means scanning it first |

No append/prepend — a hash table has no ordering to insert relative to.

In [108]:
# Test cases for SimpleDict.
# Note: for small ints, hash(x) == x, so with capacity=8, keys 1, 9, 17 all
# map to the same bucket (1 % 8 == 9 % 8 == 17 % 8 == 1) -- a guaranteed,
# reproducible collision to test chaining against.

d = SimpleDict(capacity=8)

d[1] = 'a'
d[9] = 'b'   # collides with 1
d[17] = 'c'  # collides with 1 and 9
print(d[1], d[9], d[17])   # a b c
print(len(d))              # 3

d[1] = 'z'   # update, not a duplicate insert
print(d[1], len(d))        # z 3

print(9 in d, 100 in d)    # True False

del d[9]
print(9 in d, len(d))      # False 2

print(d.get(100, 'default'))  # default
print(d.get(17))              # c

print(sorted(d.keys()))    # [1, 17]
print(sorted(d.values()))  # ['c', 'z']
print(sorted(d.items()))   # [(1, 'z'), (17, 'c')]

d.update({100: 'x', 200: 'y'})
print(len(d))               # 4
print(100 in d, d[200])     # True y


a b c
3
z 3
True False
False 2
default
c
[1, 17]
['c', 'z']
[(1, 'z'), (17, 'c')]
4
True y


## `__xxx__` vs `_xxx` — two different things that look similar

`SimpleDict` uses both: `__setitem__`, `__getitem__`, etc. (double underscore, both sides) vs. `_hash_index`, `_buckets`, `_length`, `_capacity` (single leading underscore). They mean very different things.

- **`__xxx__` (dunder/magic methods)** — a fixed, Python-recognized set of names tied to specific language behavior (see `oop_basics.ipynb`). The interpreter itself looks for these: `d[key] = value` calls `d.__setitem__(key, value)`, `len(d)` calls `d.__len__()`, `key in d` calls `d.__contains__(key)`. This is real interpreter machinery — you can't invent your own dunder name and expect anything to call it automatically.

- **`_xxx` (single leading underscore)** — pure convention, enforced by nobody. It signals "internal implementation detail — not part of this class's public API, don't rely on it from outside." Python does nothing special with it: `d._hash_index('a')` works fine from outside the class. It's a documentation-by-naming convention, not a language feature.

In `SimpleDict`: the dunders (`__setitem__`, `__getitem__`, `__contains__`, `__len__`, `__delitem__`) plus `get`/`keys`/`values`/`items`/`update` are the **public mapping protocol** — what a caller is meant to use. `_hash_index`, `_buckets`, `_length`, `_capacity` are **internal state/helpers** — implementation details a caller shouldn't need to touch directly.

(Separate gotcha worth knowing: a **double leading underscore with no trailing underscore**, e.g. `__foo`, triggers *name mangling* — Python rewrites it to `_ClassName__foo` internally, mainly to avoid subclass attribute clashes. That's a different mechanism from both of the above and doesn't apply to any of our names here, since ours either have trailing underscores too (dunders) or just one leading underscore.)

## Are dicts ordered?

- **Since Python 3.7, plain `dict` is guaranteed to preserve insertion order** — iterating a dict, or its `.keys()`/`.values()`/`.items()`, gives items back in the order they were inserted. This is a real language guarantee (it was a CPython implementation detail in 3.6, made official in 3.7).
- **`collections.OrderedDict`** is a separate, older class (predates the 3.7 guarantee) that also preserves insertion order, plus a couple of extras plain `dict` doesn't have: `.move_to_end(key)`, and equality comparison that's order-sensitive (two `OrderedDict`s with the same pairs in different orders are `!=`, whereas the plain-dict equivalent is `==`).
- **How this squares with "keys don't live in any particular order in memory" (above):** that's true of the raw hash-bucket layout — which key hashes to which slot is arbitrary. But modern CPython's `dict` keeps a *separate* insertion-ordered array alongside the hash table, and iteration walks that array, not the hash buckets directly. So the hash-based storage is still unordered internally, but the dict as a whole guarantees ordered iteration via that extra bookkeeping. Our `SimpleDict` doesn't do this — iterating its buckets gives hash order, not insertion order.

In [ ]:
# TODO: First recurring character.
# Given a string, find the first character that repeats -- scanning left to
# right, return the character you encounter that you've already seen earlier
# in the string. Return None if there's no such character.
#
# Example: "acbbac" -> 'b' (repeats at index 3, before 'a' repeats at index 4)

# Time complexity: O(n) -- one pass, O(1) average per set lookup/insert
# Space complexity: O(n) -- worst case (no repeats) stores every character
def first_recurring_char(string):
    seen = set([])
    for s in string:
        if s in seen: 
            return s
        else:
            seen.add(s)  # how to add to a set
    return None

a = 'abcdse'
print(first_recurring_char(a))

In [ ]:
# Interview problem: Two Sum
#
# Given an array of integers nums and an integer target, return the
# indices i and j such that nums[i] + nums[j] == target and i != j.
#
# You may assume that every input has exactly one pair of indices i and j
# that satisfy the condition.
#
# Return the answer with the smaller index first.

from typing import List

class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:

        map_ = {}
        for i, b in enumerate(nums):
            if b in map_:
                map_[b].append(i)
            else:
                map_[b] = [i]
        
        for a in map_:
            if target - a in map_:
                js_ = map_[target-a]
                is_ = map_[a]
                if a == target - a:
                    if len(is_) > 1:
                        return [is_[0], is_[1]]
                else:
                    return[is_[0], js_[0]]


In [ ]:
# Interview problem: Valid Palindrome
#
# Given a string s, return true if it is a palindrome, otherwise return
# false.
#
# A palindrome is a string that reads the same forward and backward. It
# is also case-insensitive and ignores all non-alphanumeric characters.
#
# Note: Alphanumeric characters consist of letters (A-Z, a-z) and
# numbers (0-9).

class Solution:
    def isPalindrome(self, s: str) -> bool:
        n = len(s)
        if n == 0 or n==1:
            return True
        
        back_indx = n - 1
        for_indx = 0
        while for_indx < back_indx:
            
            s_for = s[for_indx]
            s_back = s[back_indx]

            if not s_for.isalnum():
                for_indx += 1
                
            elif not s_back.isalnum():
                back_indx -= 1

            else:
                if s_for.lower() != s_back.lower():
                    return False
                
                for_indx += 1
                back_indx -= 1
                    
        return True

s = 'a,,,,,bZ,,Ya'
Solution().isPalindrome(s)


0 11 a a
1 10 , Y
2 10 , Y
3 10 , Y
4 10 , Y
5 10 , Y
6 10 b Y


False

In [ ]:
# Interview problem: 3Sum
#
# Given an integer array nums, return all the triplets [nums[i], nums[j], nums[k]]
# where nums[i] + nums[j] + nums[k] == 0, and the indices i, j and k are all
# distinct.
#
# The output should not contain any duplicate triplets. You may return the
# output and the triplets in any order.

#### this is the first implementation with minimal change to my two-sum solution!! 
### need to improve this code + add the two-pointer based implementation 

from typing import List 
class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:

        map_ = {}
        for i, b in enumerate(nums):
            if b in map_:
                map_[b].append(i)
            else:
                map_[b] = [i]
        
        ps = []
        for a in map_:
            if target - a in map_:
                js_ = map_[target-a]
                is_ = map_[a]
                if a == target - a:
                    if len(is_) > 1:
                        ps.append([nums[is_[0]], nums[is_[1]], -target])
                else:
                    ps.append([nums[is_[0]], nums[js_[0]], -target])
        return ps

    def RemoveDup(self, lists):
        s = set()
        for i in lists:
            b = tuple(sorted(tuple(i)))
            if b not in s:
                s.add(b)
        return([list(a) for a in s])

    
    def threeSum(self, nums: List[int]) -> List[List[int]]:
        s = []
        for i, e in enumerate(nums):
            rest = nums[0:i] + nums[i+1:]
            ps = self.twoSum(rest, -1 * e)
            if ps:
                s += ps
        
        return self.RemoveDup(s)
        
nums=[-1,0,1,2,-1,-4,-2,-3,3,0,4]
print(Solution().threeSum(nums))

[[-3, -1, 4], [-1, 0, 1], [-4, 1, 3], [-1, -1, 2], [-3, 1, 2], [-3, 0, 3], [-4, 0, 4], [-2, -1, 3], [-2, 0, 2]]


## `list` vs. `tuple` vs. `set`

A **tuple** is an ordered, fixed-size collection, written `(1, 2, 3)` — like a list, but immutable: once created, you can't add, remove, or reassign elements. It's created with `()` (or just a comma, e.g. `1, 2` — the parens aren't what makes it a tuple).

| | `list` | `tuple` | `set` |
|---|---|---|---|
| Ordered? | Yes (insertion order) | Yes (insertion order) | No |
| Mutable? | Yes | No (shallowly — see above) | Yes (but elements must be hashable) |
| Duplicates allowed? | Yes | Yes | No — duplicates collapse automatically |
| Indexable (`x[i]`)? | Yes | Yes | No — no positional access |
| Backed by | Dynamic array | Dynamic array | Hash table |
| Membership check (`x in ...`) | O(n) | O(n) | O(1) average |
| Hashable itself (usable as a dict key)? | No | Yes, if all elements are hashable | No |

- Use a **list** when order matters and you need to mutate/index into a sequence.
- Use a **tuple** for fixed, order-matters data that shouldn't change — e.g. coordinates, or as a dict key/set element (lists can't be either, since they're unhashable).
- Use a **set** when you only care about uniqueness/membership, not order or position — see "Sets vs. dicts" above for more on when a set beats a dict.

In [ ]:
# Interview problem: Best Time to Buy and Sell Stock
#
# You are given an integer array prices where prices[i] is the price of
# NeetCoin on the ith day.
#
# You may choose a single day to buy one NeetCoin and choose a different
# day in the future to sell it.
#
# Return the maximum profit you can achieve. You may choose to not make
# any transactions, in which case the profit would be 0.

class Solution:
    def maxProfit(self, prices: List[int]) -> int:
       
        temp_buy_price = float('inf')
        buy_price = float('inf')
        sell_price = - float('inf')

        for i in range(len(prices) - 1):
            if prices[i] <= temp_buy_price:
                temp_buy_price = prices[i]
                
            if prices[i + 1] - temp_buy_price > sell_price - buy_price:
                buy_price = temp_buy_price
                sell_price = prices[i + 1]
                
        return sell_price - buy_price if sell_price - buy_price > 0 else 0